# Who Gets Heard?
*A Human-Centered Dara Study of Community Reporting in Seattle.*

## Research Scope
This study examines reporting channel patterns in Seattle's 2025 Customer Service Request data. It focuses on *Find It Fix It* App and Phone reports across census tracts and explores their associations with neighborhood median household income.

## 1. Data Collection and Quality Check

### 1.1 ACS Population and Income Data
Retrieve the 2020-2024 ACS dara and check for missing values and duplicate census tract IDs.

In [1]:
import pandas as pd
import requests

# ACS dataset API
acs_url = (
    "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/"
    "arcgis/rest/services/CensusTractTop50_5YALL/"
    "FeatureServer/0/query"
)

# Retrieve 2020–2024 ACS data
params = {
    "where": "ACS_VINTAGE='5Y24'",
    "outFields": (
        "GEOID,ACS_VINTAGE,"
        "TOTAL_POPULATION,"
        "MEDIAN_HH_INC_PAST_12MO_DOLLAR"
    ),
    "returnGeometry": "false",
    "resultRecordCount": 2000,
    "f": "json"
}

response = requests.get(acs_url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

if "error" in data:
    raise RuntimeError(data["error"])

if data.get("exceededTransferLimit"):
    raise RuntimeError("The API did not return all records.")

acs = pd.DataFrame(
    [feature["attributes"] for feature in data["features"]]
)

print("Number of records:", len(acs))

print("\nMissing values:")
print(acs.isna().sum())

print("\nDuplicate GEOIDs:")
print(acs["GEOID"].duplicated().sum())

display(acs.head())

Number of records: 494

Missing values:
GEOID                              0
ACS_VINTAGE                        0
TOTAL_POPULATION                   0
MEDIAN_HH_INC_PAST_12MO_DOLLAR    21
dtype: int64

Duplicate GEOIDs:
0


,GEOID,ACS_VINTAGE,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
0,53033000402,5Y24,5365,128310.0
1,53033000403,5Y24,3285,65469.0
2,53033000700,5Y24,4857,95431.0
3,53033003302,5Y24,3826,153523.0
4,53033003601,5Y24,3507,111908.0


In [2]:
# Check population and income values
display(
    acs[
        ["TOTAL_POPULATION",
         "MEDIAN_HH_INC_PAST_12MO_DOLLAR"]
    ].describe()
)

# Check GEOID format
acs["GEOID"] = acs["GEOID"].astype(str)

print("GEOID lengths:")
print(acs["GEOID"].str.len().value_counts())

# Check records with missing income
print("\nRecords with missing income:")
display(
    acs.loc[
        acs["MEDIAN_HH_INC_PAST_12MO_DOLLAR"].isna(),
        ["GEOID", "TOTAL_POPULATION",
         "MEDIAN_HH_INC_PAST_12MO_DOLLAR"]
    ].head(10)
)

,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
count,494.000000,473.000000
mean,4629.900810,130985.822410
std,1296.307837,45572.715752
min,1295.000000,17500.000000
25%,3680.000000,98478.000000
50%,4468.500000,126786.000000
75%,5545.500000,160049.000000
max,8777.000000,249456.000000


GEOID lengths:
GEOID
11    494
Name: count, dtype: int64

Records with missing income:


,GEOID,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
61,53033011802,5167,NaN
106,53033032318,6767,NaN
110,53033032603,4171,NaN
162,53033006200,4193,NaN
210,53033024500,4966,NaN
239,53033032221,6173,NaN
241,53033032321,6925,NaN
250,53033025008,2533,NaN
271,53033032215,5922,NaN
273,53033032320,4474,NaN


### 1.2 Geographic Coverage Check
Compare ACS census tract IDs with Seattle's 2020 census tract boundaries to identify records relevant to the study area.

In [5]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import geopandas as gpd

# Load Seattle 2020 census tract boundaries
tract_url = (
    "https://data-seattlecitygis.opendata.arcgis.com/"
    "api/download/v1/items/9075e8c912a24c4b9458af8866c72ae7/"
    "geojson?layers=3"
)

tracts = gpd.read_file(tract_url)

# Ensure GEOIDs have the same format
tracts["GEOID20"] = tracts["GEOID20"].astype(str)
acs["GEOID"] = acs["GEOID"].astype(str)

# Check matching GEOIDs
matched = tracts["GEOID20"].isin(acs["GEOID"])

print("Seattle census tracts:", len(tracts))
print("Matching ACS records:", matched.sum())
print("Unmatched Seattle tracts:", (~matched).sum())

# Preview unmatched tracts
display(tracts.loc[~matched, ["GEOID20"]].head())

Seattle census tracts: 177
Matching ACS records: 177
Unmatched Seattle tracts: 0


,GEOID20


### 1.3 Seattle ACS Data
Keep only ACS records that match Seattle census tracts and check missing population and income values.

In [7]:
# Keep ACS records that match Seattle census tracts
seattle_acs = acs[
    acs["GEOID"].isin(tracts["GEOID20"])
].copy()

print("Seattle ACS records:", len(seattle_acs))

print("\nMissing values:")
print(seattle_acs.isna().sum())

print("\nDuplicate GEOIDs:")
print(seattle_acs["GEOID"].duplicated().sum())

display(seattle_acs.head())

Seattle ACS records: 177

Missing values:
GEOID                             0
ACS_VINTAGE                       0
TOTAL_POPULATION                  0
MEDIAN_HH_INC_PAST_12MO_DOLLAR    3
dtype: int64

Duplicate GEOIDs:
0


,GEOID,ACS_VINTAGE,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
0,53033000402,5Y24,5365,128310.0
1,53033000403,5Y24,3285,65469.0
2,53033000700,5Y24,4857,95431.0
3,53033003302,5Y24,3826,153523.0
4,53033003601,5Y24,3507,111908.0


### 1.4 Check Invalid Values
Check whether population and median household income contain zero or negative values before saving the ACS data.

In [9]:
# Check invalid population values
print("Population <= 0:",
      (seattle_acs["TOTAL_POPULATION"] <= 0).sum())

# Check invalid income values
print("Income <= 0:",
      (seattle_acs["MEDIAN_HH_INC_PAST_12MO_DOLLAR"] <= 0).sum())

# Show census tracts with missing income
print("\nCensus tracts with missing income:")
display(
    seattle_acs.loc[
        seattle_acs["MEDIAN_HH_INC_PAST_12MO_DOLLAR"].isna(),
        ["GEOID", "TOTAL_POPULATION",
         "MEDIAN_HH_INC_PAST_12MO_DOLLAR"]
    ]
)

Population <= 0: 0
Income <= 0: 0

Census tracts with missing income:


,GEOID,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
61,53033011802,5167,NaN
162,53033006200,4193,NaN
352,53033004101,3999,NaN


### 1.5 Save ACS Data
Save the ACS data from Seattle's 177 census tracts. Keep missing income values for later analysis.

In [10]:
from pathlib import Path

# Create data folder
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Select fields needed for analysis
acs_final = seattle_acs[
    [
        "GEOID",
        "ACS_VINTAGE",
        "TOTAL_POPULATION",
        "MEDIAN_HH_INC_PAST_12MO_DOLLAR"
    ]
].copy()

# Save as CSV
acs_final.to_csv(
    data_dir / "acs_2024.csv",
    index=False
)

print("Saved:", data_dir / "acs_2024.csv")
print("Number of records:", len(acs_final))

Saved: data/acs_2024.csv
Number of records: 177


### 1.6 Retrieve 2025 Service Requests
Connect to the Seattle Customer Service Request API and check the total number of records and available fields for 2025.

In [1]:
import requests
import pandas as pd

# Seattle Customer Service Requests API
requests_url = "https://cos-data.seattle.gov/resource/5ngg-rpne.json"

# Filter records from 2025
date_filter = (
    "createddate >= '2025-01-01T00:00:00' "
    "AND createddate < '2026-01-01T00:00:00'"
)

# Count records
count_params = {
    "$where": date_filter,
    "$select": "count(*) as total"
}

response = requests.get(
    requests_url, params=count_params, timeout=60
)
response.raise_for_status()

count_data = response.json()
total = int(count_data[0]["total"])

print("Total 2025 service requests:", total)

# Retrieve five sample records
sample_params = {
    "$where": date_filter,
    "$limit": 5
}

response = requests.get(
    requests_url, params=sample_params, timeout=60
)
response.raise_for_status()

sample = pd.DataFrame(response.json())

print("\nAvailable columns:")
print(sample.columns.tolist())

display(sample.head())

Total 2025 service requests: 337937

Available columns:
['servicerequestnumber', 'webintakeservicerequests', 'departmentname', 'createddate', 'methodreceivedname', 'servicerequeststatusname', 'location', 'x_value', 'y_value', 'latitude', 'longitude', 'latitude_longitude', 'zipcode', 'councildistrict', 'policeprecinct', 'community_reporting_area']


,servicerequestnumber,webintakeservicerequests,departmentname,createddate,methodreceivedname,servicerequeststatusname,location,x_value,y_value,latitude,longitude,latitude_longitude,zipcode,councildistrict,policeprecinct,community_reporting_area
0,25-00000001,General Inquiry - Transportation,SDOT-Seattle Department of Transportation,2025-01-01T00:01:37.000,Citizen Web Intake App,Closed,"4014 UNIVERSITY WAY NE, SEATTLE, WA 98105",1275780.6382456100,242725.5219293140,47.65581185,-122.31297132,"{'type': 'Point', 'coordinates': [-122.3129713...",98105,4,NORTH,UNIVERSITY DISTRICT
1,25-00000002,Unauthorized Encampment,SEA-City of Seattle,2025-01-01T00:09:36.000,Find It Fix It Apps,Closed,"12540 33RD AVE NE, SEATTLE, WA 98125",1281410.4862134500,266333.9536218140,47.72081678,-122.29194729,"{'type': 'Point', 'coordinates': [-122.2919472...",98125,5,NORTH,CEDAR PARK/MEADOWBROOK
2,25-00000003,Parking Enforcement,SPD-Seattle Police Department,2025-01-01T00:49:55.000,Find It Fix It Apps,Closed,"2558 DEXTER AVE N, SEATTLE, WA 98109",1267656.2342126600,238464.8148629020,47.64370045,-122.34559083,"{'type': 'Point', 'coordinates': [-122.3455908...",98109,7,WEST,QUEEN ANNE
3,25-00000004,Unauthorized Encampment,SEA-City of Seattle,2025-01-01T00:51:58.000,Find It Fix It Apps,Closed,"7269 24TH AVE SW, SEATTLE, WA 98106",1262532.2079512400,199864.7845326730,47.53762187,-122.36326032,"{'type': 'Point', 'coordinates': [-122.3632603...",98106,1,SOUTHWEST,HIGH POINT
4,25-00000005,Overgrown Vegetation,FAS-Finance and Administrative Services,2025-01-01T01:18:09.000,Find It Fix It Apps,Closed,"1505 LAKEVIEW BLVD E, SEATTLE, WA 98102",1273132.6443166800,234406.0628350440,47.63286836,-122.32306038,"{'type': 'Point', 'coordinates': [-122.3230603...",98102,3,EAST,CASCADE/EASTLAKE


### 1.7 Reporting Channel Distribution
Count the number of service requests submitted through each reporting channel in 2025 to identify the channels used for comparison.

In [2]:
# Count 2025 requests by reporting channel
channel_params = {
    "$where": date_filter,
    "$select": "methodreceivedname, count(*) as count",
    "$group": "methodreceivedname",
    "$order": "count DESC",
    "$limit": 100
}

response = requests.get(
    requests_url,
    params=channel_params,
    timeout=60
)
response.raise_for_status()

channel_data = pd.DataFrame(response.json())

channel_data["count"] = channel_data["count"].astype(int)

channel_data["percentage"] = (
    channel_data["count"] / total * 100
).round(2)

display(channel_data)

,methodreceivedname,count,percentage
0,Find It Fix It Apps,268094,79.33
1,Citizen Web Intake App,36487,10.80
2,PremierOne CSR Mobile Apps,12990,3.84
3,Phone,10416,3.08
4,E-Mail,5054,1.50
5,Mass Entry,2873,0.85
6,Voice Mail,1980,0.59
7,Mail,30,0.01
8,Kiosk,3,0.00
9,Other,3,0.00


### 1.8 Request Type Distribution
Compare requesy tyoes submitted through Find It Fix Apps and Phone to identify categories suitable foor further analysis.

In [3]:
# Count request types for App and Phone
type_params = {
    "$where": (
        date_filter
        + " AND methodreceivedname IN "
        + "('Find It Fix It Apps', 'Phone')"
    ),
    "$select": (
        "methodreceivedname, "
        "webintakeservicerequests, "
        "count(*) as count"
    ),
    "$group": (
        "methodreceivedname, "
        "webintakeservicerequests"
    ),
    "$limit": 5000
}

response = requests.get(
    requests_url,
    params=type_params,
    timeout=60
)
response.raise_for_status()

type_data = pd.DataFrame(response.json())
type_data["count"] = type_data["count"].astype(int)

# Compare App and Phone within each request type
type_comparison = type_data.pivot_table(
    index="webintakeservicerequests",
    columns="methodreceivedname",
    values="count",
    aggfunc="sum",
    fill_value=0
)

type_comparison["Total"] = type_comparison.sum(axis=1)

type_comparison = type_comparison.sort_values(
    "Total", ascending=False
)

display(type_comparison.head(20))

methodreceivedname,Find It Fix It Apps,Phone,Total
webintakeservicerequests,,,
Unauthorized Encampment,64192,163,64355
Abandoned Vehicle/72hr Parking Ordinance,51002,4067,55069
Illegal Dumping / Needles,26579,372,26951
Parking Enforcement,22715,31,22746
Graffiti,21726,376,22102
General Inquiry - Customer Service Bureau,14081,1745,15826
Parks and Recreation Maintenance,12288,98,12386
Scooter or Bike Share Issue,8711,419,9130
Pothole,8205,12,8217


### 1.9 Missing Values and Duplicate Records
Check missing geographic coordinate, request types, and duplicate request IDs for App and Phone reports in 2025.

In [4]:
# Check missing values and duplicate IDs by channel
quality_params = {
    "$where": (
        date_filter
        + " AND methodreceivedname IN "
        + "('Find It Fix It Apps', 'Phone')"
    ),
    "$select": (
        "methodreceivedname, "
        "count(*) as total, "
        "count(latitude) as with_latitude, "
        "count(longitude) as with_longitude, "
        "count(webintakeservicerequests) as with_type, "
        "count(distinct servicerequestnumber) as unique_ids"
    ),
    "$group": "methodreceivedname"
}

response = requests.get(
    requests_url,
    params=quality_params,
    timeout=60
)
response.raise_for_status()

quality_data = pd.DataFrame(response.json())

numeric_cols = [
    "total", "with_latitude", "with_longitude",
    "with_type", "unique_ids"
]

quality_data[numeric_cols] = quality_data[numeric_cols].astype(int)

quality_data["missing_latitude"] = (
    quality_data["total"] - quality_data["with_latitude"]
)

quality_data["missing_longitude"] = (
    quality_data["total"] - quality_data["with_longitude"]
)

quality_data["missing_type"] = (
    quality_data["total"] - quality_data["with_type"]
)

quality_data["duplicate_ids"] = (
    quality_data["total"] - quality_data["unique_ids"]
)

display(quality_data[
    [
        "methodreceivedname",
        "total",
        "missing_latitude",
        "missing_longitude",
        "missing_type",
        "duplicate_ids"
    ]
])

,methodreceivedname,total,missing_latitude,missing_longitude,missing_type,duplicate_ids
0,Find It Fix It Apps,268094,58,58,0,0
1,Phone,10416,710,710,0,0


### 1.10 Retrieve and Save Service Requests
Retrieve all 2025 Find It Fix Ir App and Phone reports in batches and save the records locally for further analysis.

In [5]:
from pathlib import Path
import time

# Select the fields needed for analysis
fields = [
    "servicerequestnumber",
    "createddate",
    "methodreceivedname",
    "webintakeservicerequests",
    "latitude",
    "longitude"
]

channel_filter = (
    date_filter
    + " AND methodreceivedname IN "
    + "('Find It Fix It Apps', 'Phone')"
)

batch_size = 10000
all_records = []

for offset in range(0, 278510, batch_size):
    params = {
        "$where": channel_filter,
        "$select": ",".join(fields),
        "$order": "servicerequestnumber ASC",
        "$limit": batch_size,
        "$offset": offset
    }

    response = requests.get(
        requests_url, params=params, timeout=120
    )
    response.raise_for_status()

    batch = response.json()
    all_records.extend(batch)

    print(f"Retrieved {len(all_records)} records")
    time.sleep(0.2)

requests_2025 = pd.DataFrame(all_records)

print("\nTotal downloaded:", len(requests_2025))
print("Unique IDs:",
      requests_2025["servicerequestnumber"].nunique())

# Save the downloaded records
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

requests_2025.to_csv(
    data_dir / "seattle_requests_2025.csv",
    index=False
)

print("Saved: data/seattle_requests_2025.csv")

Retrieved 10000 records
Retrieved 20000 records
Retrieved 30000 records
Retrieved 40000 records
Retrieved 50000 records
Retrieved 60000 records
Retrieved 70000 records
Retrieved 80000 records
Retrieved 90000 records
Retrieved 100000 records
Retrieved 110000 records
Retrieved 120000 records
Retrieved 130000 records
Retrieved 140000 records
Retrieved 150000 records
Retrieved 160000 records
Retrieved 170000 records
Retrieved 180000 records
Retrieved 190000 records
Retrieved 200000 records
Retrieved 210000 records
Retrieved 220000 records
Retrieved 230000 records
Retrieved 240000 records
Retrieved 250000 records
Retrieved 260000 records
Retrieved 270000 records
Retrieved 278510 records

Total downloaded: 278510
Unique IDs: 278510
Saved: data/seattle_requests_2025.csv


### 1.11 Save Census Tract Boundaries
Save Seattle's 2020 census tract boundaries as a GeoJSON file for geographic analysis.

In [14]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [15]:
import geopandas as gpd

tract_url = (
    "https://data-seattlecitygis.opendata.arcgis.com/"
    "api/download/v1/items/9075e8c912a24c4b9458af8866c72ae7/"
    "geojson?layers=3"
)

tracts = gpd.read_file(tract_url)

print("Number of census tracts:", len(tracts))
print("Coordinate system:", tracts.crs)

Number of census tracts: 177
Coordinate system: EPSG:2926


In [18]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

tracts.to_file(
    data_dir / "tracts_2020.geojson",
    driver="GeoJSON"
)

print("Saved: data/tracts_2020.geojson")
print("Number of census tracts:", len(tracts))

Saved: data/tracts_2020.geojson
Number of census tracts: 177


### 1.12 Data Preparation Summary
I collected three datasets for this study:

- **2025 Seattle Customer Service Requests:** 278510 records from Find It Fix It Apps and Phone, with no duplicate request IDS. Some reports have missing geographic coordinates, especially Phone reports.
- **2020 - 2024 ACS Data:** Population and median household income data for 177 Seattle census tracts. Three tracts have missing income values, which will be excluded only from income-related analysis.
- **2020 Census Tract Boundaries:** Geographic boundaries for 177 Seattle census tracts. All 177 GEOIDs matched the ACS dataset.

The datasets were saved locally for further analysis. Records with missing coordinates will be retained in the original dataset but excluded from spatial analysis when necessary.